# Facility Recommender — Guided Tour

This project answers one question: **given a patient's symptoms and location, which nearby facilities can actually handle it — not just claim to?**

The scraped facility directory is marketing copy — any clinic can list "cardiology" on its page. The pipeline built here cross-checks every claimed service against what the facility also shows (equipment, procedures, capabilities) and scores each facility × service pair into **confidence bands** (high / medium / low / unverified). The recommender then turns symptoms into a needs vector, filters facilities by geography, ranks by verified capability, and explains every recommendation.

Run the cells top to bottom. The demo cell at the bottom takes ~30s (one LLM call to extract needs from the symptom text).

## The data, in plain terms

All tables live in the `labuser_jdw2` catalog:

| Table | What it is |
| --- | --- |
| `gold.facility_district_health_indicators` | One row per facility (**10,077**): name, coordinates, facility type, raw scraped evidence columns, plus NFHS-5 district health context (96.2% of facilities matched to their district) |
| `platinum.service_catalog` | **192 canonical services** — 101 specialties (cardiology, pediatrics, …) + 91 concrete services (dialysis, MRI scanner, ICU, root canal, …) |
| `platinum.facility_service_evidence` | **277,063 rows**: for every (facility, service), which *channel* mentions it — claimed specialty, procedures, equipment, capabilities, or description text |
| `platinum.corroboration_rules` | Per service, what should back it up: e.g. CABG surgery needs ICU + operation theatre + blood bank; neurosurgery needs hospital class + 3+ doctors |
| `platinum.facility_service_confidence` | **200,226 (facility, service) pairs** with a trust band + full explainability (channels, missing requirements, scale checks) |

The key idea: **a claim alone is weak, not wrong** (unknown ≠ no). Multiple independent channels corroborating each other → `high`. A claim that contradicts the facility's scale (a 2-doctor clinic claiming neurosurgery) → `low`.

In [0]:
display(spark.sql("""
  SELECT service_type, COUNT(*) AS n_services
  FROM labuser_jdw2.platinum.service_catalog
  GROUP BY 1 ORDER BY 2 DESC
"""))

display(spark.sql("""
  SELECT confidence_band, COUNT(*) AS n_pairs
  FROM labuser_jdw2.platinum.facility_service_confidence
  GROUP BY 1 ORDER BY 2 DESC
"""))

In [0]:
# What a 'high' band looks like: a service backed by multiple channels,
# with its corroboration requirements met (required_missing is empty).
display(spark.sql("""
  SELECT unique_id, service, confidence_band,
         channels, required_missing, scale_note
  FROM labuser_jdw2.platinum.facility_service_confidence
  WHERE service = 'dialysis' AND confidence_band = 'high'
  LIMIT 5
"""))

In [0]:
# The end-goal demo: symptoms + pincode -> ranked facilities with reasons.
# ~30s: one ai_query call (gpt-oss-120b, temperature 0) extracts the needs vector,
# constrained to the 192-service catalog.
import runpy

rec = runpy.run_path("/Workspace/Users/jdw2@andrew.cmu.edu/cmu_datathon26/src/recommend.py")

lat, lng = rec["pincode_to_coords"](spark, 143001)  # Amritsar
needs, ranked = rec["recommend"](
    spark, lat, lng, 25.0,                      # 25 km radius
    "chest pain and shortness of breath, sweating",
    "58-year-old male with hypertension")

rows = ranked.collect()
for f in rec["explain"](spark, needs, rows)[:3]:
    print(rec["render_explanation"](f), "\n")

# Try your own: change the pincode, radius, symptoms, or context above.
# e.g. "severe knee pain, doctor advised knee replacement surgery",
#      "child with high fever and rash", "trying to conceive for three years".

## How to read the demo output

* **Score** = Σ (need weight × confidence score), where high = 1.0 / medium = 0.5 / unverified = 0.3 / low = 0.05.
* **Critical needs are hard filters** — a facility that can't credibly provide them (missing evidence, or band `low`) is never recommended. For chest pain, `emergencyCare` is critical.
* Each line of an explanation cites **only verified evidence**: the band, the channels that corroborated it (e.g. `capability+equipment+procedure`), token counts, and scale checks. "No evidence found for: X (unknown, not disproven)" is honest — never treated as a disqualification.
* Tie-breakers: NABH accreditation → bed count → facility class → distance.

**Caveat (the honest part):** confidence measures *internal consistency* of the facility's own scraped description — it is **not** clinical outcome quality. See `cmu_datathon26/QA_REPORT.md` for the full coverage stats, contradiction rates, and nine explicit blind spots.